# Key parameters: the evidence behind each value

This notebook re-derives the tables in `reference/KEY_PARAMETERS.md`
from the frozen record in `reference/research_record/` and from the
price store. It writes nothing. Run it after any research pass so the
document and the record cannot drift apart.

Sections follow the document: ground truth, candidacy, features,
model and selection, operating point, inflection, display, bot screen.

In [1]:
from __future__ import annotations

import json
import os
import sys

import numpy as np
import pandas as pd

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(
    globals().get("__file__", os.path.join(os.getcwd(), "x")))))
if not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    ROOT = os.getcwd()
sys.path.insert(0, ROOT)

from src import config as C                                   # noqa: E402

RECORD = os.path.join(ROOT, "reference", "research_record")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


def rec(name):
    """Load one frozen JSON record by stem."""
    with open(os.path.join(RECORD, name + ".json"), encoding="utf-8") as f:
        return json.load(f)

## 1. Ground truth

The frozen bars, as `src/config.py` holds them.

In [2]:
pd.Series({
    "EUPHORIA_BOOM_MIN_ETF": C.EUPHORIA_BOOM_MIN_ETF,
    "EUPHORIA_BOOM_MIN_SINGLE": C.EUPHORIA_BOOM_MIN_SINGLE,
    "EUPHORIA_CRASH_MIN_ETF": C.EUPHORIA_CRASH_MIN_ETF,
    "EUPHORIA_CRASH_MIN_SINGLE": C.EUPHORIA_CRASH_MIN_SINGLE,
    "EUPHORIA_BOOM_LOOKBACK_D": C.EUPHORIA_BOOM_LOOKBACK_D,
    "EUPHORIA_CRASH_WINDOW_D": C.EUPHORIA_CRASH_WINDOW_D,
    "EUPHORIA_PEAK_LOCAL_MAX_D": C.EUPHORIA_PEAK_LOCAL_MAX_D,
    "EUPHORIA_PEAK_MERGE_D": C.EUPHORIA_PEAK_MERGE_D,
}, name="value")

EUPHORIA_BOOM_MIN_ETF          0.20
EUPHORIA_BOOM_MIN_SINGLE       0.40
EUPHORIA_CRASH_MIN_ETF         0.12
EUPHORIA_CRASH_MIN_SINGLE      0.25
EUPHORIA_BOOM_LOOKBACK_D     120.00
EUPHORIA_CRASH_WINDOW_D       90.00
EUPHORIA_PEAK_LOCAL_MAX_D     21.00
EUPHORIA_PEAK_MERGE_D         30.00
Name: value, dtype: float64

### 1a. The bar sweep

All four bars scaled together. The adopted row is the 0.8x setting
(0.20 / 0.12 / 0.40 / 0.24, with the single-name bust bar rounded to
0.25 in config).

In [3]:
gt = pd.DataFrame(rec("sweep_ground_truth")["rows"])
gt[["setting", "boom_etf", "crash_etf", "boom_single", "crash_single",
    "peaks", "detectable", "captured", "rate_of_detectable", "fa_per_iy",
    "instruments_with_any_episode"]]

,setting,boom_etf,crash_etf,boom_single,crash_single,peaks,detectable,captured,rate_of_detectable,fa_per_iy,instruments_with_any_episode
0,tighter (1.4x),0.350,0.210,0.70,0.42,102,45,10,0.222,0.23,26
1,tighter (1.2x),0.300,0.180,0.60,0.36,176,72,14,0.194,0.22,36
2,previous (1.0x),0.250,0.150,0.50,0.30,282,115,18,0.157,0.21,43
3,looser (0.8x),0.200,0.120,0.40,0.24,490,214,25,0.117,0.19,52
4,looser (0.6x),0.150,0.090,0.30,0.18,858,368,35,0.095,0.17,58
5,looser (0.5x),0.125,0.075,0.25,0.15,1167,501,43,0.086,0.14,58
6,crash-only looser,0.250,0.100,0.50,0.20,450,193,25,0.130,0.19,52
7,boom-only looser,0.150,0.150,0.30,0.30,460,196,19,0.097,0.20,54


### 1b. The lookback sweep

In [4]:
pd.DataFrame(rec("sweep_lookback")["lookback_sweep"])

,lookback_d,peaks,detectable,captured,rate,fa_per_iy,anchors_kept
0,54,0,0,0,NaN,0.26,0
1,90,229,93,18,0.194,0.21,4
2,120,282,115,18,0.157,0.21,4
3,180,330,132,19,0.144,0.20,4
4,250,392,159,19,0.119,0.20,4
5,365,457,201,19,0.095,0.20,4


### 1c. What the record holds

In [5]:
e = rec("nb01_episode_stats")
pd.Series({k: e[k] for k in ("episodes_total", "onset_detectable",
                              "top_detectable", "median_run_days",
                              "median_boom_pct", "median_bust_pct")})

episodes_total      494.000000
onset_detectable    199.000000
top_detectable      219.000000
median_run_days      93.500000
median_boom_pct       0.488839
median_bust_pct      -0.266337
dtype: float64

In [6]:
pd.DataFrame(e["per_year"]).T

,episodes,onset_detectable,top_detectable
2017,5,1,1
2018,23,15,16
2019,19,14,13
2020,56,35,37
2021,67,52,53
2022,76,42,40
2023,63,18,16
2024,65,1,1
2025,53,0,0
2026,67,21,42


### 1d. Unconditional baselines: why single names carry a higher bar

For every instrument that has at least one episode, the trailing
120-day run-up from the window low and the forward 90-day maximum
drawdown on every day of history. The bars are read against these.

In [7]:
prices_path = os.path.join(ROOT, "data", "prices", "prices.parquet")
episodes_path = os.path.join(ROOT, "data", "processed", "episodes.parquet")
if os.path.exists(prices_path) and os.path.exists(episodes_path):
    px = pd.read_parquet(prices_path)
    ep = pd.read_parquet(episodes_path)
    kinds = ep.drop_duplicates("symbol").set_index("symbol")["kind"]
    wide = (px.pivot(index="date", columns="symbol", values="px_last")
              .sort_index())
    wide = wide[[c for c in wide.columns if c in kinds.index]]
    runup = wide / wide.rolling(C.EUPHORIA_BOOM_LOOKBACK_D,
                                min_periods=60).min() - 1
    fwd_min = wide[::-1].rolling(C.EUPHORIA_CRASH_WINDOW_D,
                                 min_periods=30).min()[::-1]
    drawdown = fwd_min / wide - 1
    rows = []
    for kind, boom_bar, bust_bar in (
            ("theme", C.EUPHORIA_BOOM_MIN_ETF, C.EUPHORIA_CRASH_MIN_ETF),
            ("single", C.EUPHORIA_BOOM_MIN_SINGLE, C.EUPHORIA_CRASH_MIN_SINGLE)):
        cols = [c for c in wide.columns if kinds[c] == kind]
        r, d = runup[cols].stack(), drawdown[cols].stack()
        rows.append({
            "kind": kind, "instruments": len(cols),
            "runup_median": r.median(), "runup_p75": r.quantile(0.75),
            "share_days_over_boom_bar": (r >= boom_bar).mean(),
            "drawdown_median": d.median(),
            "share_days_under_bust_bar": (d <= -bust_bar).mean(),
            "boom_bar_over_median": boom_bar / r.median(),
            "bust_bar_over_median": bust_bar / -d.median(),
        })
    baselines = pd.DataFrame(rows).set_index("kind").round(3)
else:
    baselines = "price store or episodes not present on this copy"
baselines

,instruments,runup_median,runup_p75,share_days_over_boom_bar,drawdown_median,share_days_under_bust_bar,boom_bar_over_median,bust_bar_over_median
kind,,,,,,,,
theme,25,0.148,0.244,0.303,-0.060,0.234,1.354,2.008
single,20,0.295,0.599,0.293,-0.141,0.233,1.358,1.772


## 2. Candidacy gates

In [8]:
pd.Series({
    "EUPHORIA_MIN_COVERAGE (A0)": C.EUPHORIA_MIN_COVERAGE,
    "EUPHORIA_HYPE_MULT (A1)": C.EUPHORIA_HYPE_MULT,
    "EUPHORIA_ONSET_HYPE_MIN": C.EUPHORIA_ONSET_HYPE_MIN,
    "EUPHORIA_ATT_GATE (A2)": C.EUPHORIA_ATT_GATE,
    "EUPHORIA_BOOM_WINDOW_D": C.EUPHORIA_BOOM_WINDOW_D,
    "EUPHORIA_PCT_WINDOW": C.EUPHORIA_PCT_WINDOW,
    "EUPHORIA_MIN_HISTORY": C.EUPHORIA_MIN_HISTORY,
    "EUPHORIA_EXCLUDED_THEMES": ", ".join(sorted(C.EUPHORIA_EXCLUDED_THEMES)),
}, name="value")

EUPHORIA_MIN_COVERAGE (A0)                         100
EUPHORIA_HYPE_MULT (A1)                            2.0
EUPHORIA_ONSET_HYPE_MIN                            1.1
EUPHORIA_ATT_GATE (A2)                             0.9
EUPHORIA_BOOM_WINDOW_D                              54
EUPHORIA_PCT_WINDOW                                365
EUPHORIA_MIN_HISTORY                               180
EUPHORIA_EXCLUDED_THEMES      rates_bonds, real_estate
Name: value, dtype: object

Why each near-miss day was not called (the census), and the loosening
sweep with the shipped values marked.

In [9]:
st = rec("nb06_strictness")
pd.DataFrame({h: st["near_miss_census"][h]["by_cause"]
              for h in st["near_miss_census"]}).fillna(0).astype(int)

,get_out,get_in
1 attention gate,1381,0
2 persistence,777,0
3 crowd 2x,3413,0
4 boom gate,1344,0
5 smoothing,75,0
6 cooldown,329,0
1 entry floor 1.10x,0,19
2 end-stage phase gate,0,328
3 smoothing,0,430
4 cooldown,0,252


In [10]:
pd.DataFrame(st["loosening_sweep"]["table"])

,knob,value,shipped,captured,detectable,false_alarms,fa_per_iy,alerts,pareto
0,hype gate,1.25,False,23,98,14,0.089744,47,True
1,hype gate,1.50,False,22,98,13,0.095588,45,False
2,hype gate,1.75,False,16,98,8,0.062500,32,True
3,hype gate,2.00,True,21,98,9,0.075000,40,True
4,boom gate (ETF),0.10,False,18,98,29,0.161111,57,False
5,boom gate (ETF),0.15,False,18,98,19,0.121795,46,False
6,boom gate (ETF),0.20,False,20,98,12,0.083333,42,False
7,boom gate (ETF),0.25,True,21,98,9,0.075000,40,True
8,attention gate,0.80,False,14,98,6,0.050000,30,False
9,attention gate,0.85,False,19,98,9,0.075000,38,False


## 3. Features

Per-feature separation (walk-forward, with bootstrap intervals). Every
feature is weak alone; the bank is judged in combination.

In [11]:
fs = pd.DataFrame(rec("nb02_feature_stats"))
(fs.pivot(index="feature", columns="label", values=["auroc", "ap_lift"])
   .round(3))

auroc        ap_lift       
label               y_onset  y_top y_onset  y_top
feature                                          
attention_accel       0.518  0.507   1.152  1.095
attention_convexity   0.545  0.512   1.119  1.093
bull_inflection       0.543  0.515   1.126  1.016
e1                    0.551  0.553   1.171  1.187
e2                    0.465  0.558   0.903  1.141
e3                    0.536  0.507   1.186  1.103
e5                    0.545  0.512   1.119  1.093
fade                  0.504  0.510   1.007  1.022
hype_ratio            0.529  0.516   1.094  1.070
influx_speed          0.521  0.504   1.151  1.099

In [12]:
pd.DataFrame(rec("research_stats")["feature_correlation_spearman"]).round(2)

,e1,e2,e3,e5,fade,level,label
e1,1.00,-0.02,0.80,0.31,0.34,0.42,-0.01
e2,-0.02,1.00,-0.04,-0.12,-0.07,0.50,0.18
e3,0.80,-0.04,1.00,0.51,0.31,0.60,-0.04
e5,0.31,-0.12,0.51,1.00,0.18,0.67,0.01
fade,0.34,-0.07,0.31,0.18,1.00,0.16,-0.05
level,0.42,0.50,0.60,0.67,0.16,1.00,0.07
label,-0.01,0.18,-0.04,0.01,-0.05,0.07,1.00


## 4. Model and selection

The tournament, both heads, walk-forward test years only.

In [13]:
t = rec("ml_tournament")
print("winner:", t["results"].get("winner"))
rows = []
for head, fams in t["results"].items():
    if not isinstance(fams, dict):
        continue
    for fam, r in fams.items():
        rows.append({"head": head, "family": fam,
                     "capture_rate": r.get("capture_rate"),
                     "auroc": r.get("auroc"),
                     "ap_lift": (r["ap"] / r["ap_baseline"]
                                 if r.get("ap") and r.get("ap_baseline") else None),
                     "fa_per_iy": r.get("fa_per_iy"),
                     "precision": r.get("precision"),
                     "median_lead_days": r.get("median_lead_days")})
pd.DataFrame(rows).round(3)

winner: ens


,head,family,capture_rate,auroc,ap_lift,fa_per_iy,precision,median_lead_days
0,get_out,logit_crowd,0.243,0.572,1.192,0.677,0.141,17
1,get_out,logit,0.472,0.688,1.933,0.465,0.317,11
2,get_out,gbm_crowd,0.229,0.553,1.106,0.658,0.137,13
3,get_out,gbm,0.454,0.715,2.038,0.574,0.265,14
4,get_out,mlp_crowd,0.170,0.532,1.106,0.637,0.109,15
5,get_out,mlp,0.427,0.695,1.837,0.925,0.174,11
6,get_out,ens_crowd,0.372,0.549,1.192,1.122,0.131,16
7,get_out,ens,0.422,0.717,2.452,0.346,0.358,16
8,get_out,rules,0.117,0.549,1.131,0.094,0.595,12
9,get_in,logit_crowd,0.359,0.548,1.153,0.807,0.156,16


The production record the dashboard quotes.

In [14]:
report_path = os.path.join(ROOT, "data", "processed", "euphoria_desk_report.json")
if not os.path.exists(report_path):
    report_path = os.path.join(ROOT, "DASHBOARD_DATA", "euphoria_desk_report.json")
if os.path.exists(report_path):
    with open(report_path, encoding="utf-8") as f:
        rep = json.load(f)
    print("model:", rep.get("model"))
    print("selection rule:", rep.get("selection_rule"))
    prod = {h: {k: rep["tournament"][h][rep["model"]].get(k)
                for k in ("capture_rate", "median_lead_days", "auroc", "ap",
                          "ap_baseline", "fa_per_iy")}
            for h in rep.get("tournament", {})}
    display(pd.DataFrame(prod).T)   # noqa: F821  (notebook display)
else:
    print("no production record on this copy")

model: ens
selection rule: one family for both heads; combined test AP lift; ties AUROC then fewer false alarms


,capture_rate,median_lead_days,auroc,ap,ap_baseline,fa_per_iy
get_out,0.445,14.0,0.731,0.260,0.095,0.575
get_in,0.569,19.0,0.734,0.257,0.105,0.404


What the price features add: crowd-only vs crowd + price.

In [15]:
pb = rec("nb08_price_blind")
pd.DataFrame({k: {m: v.get(m) for m in ("auroc", "ap", "ap_baseline",
                                        "captured", "false_alarms")}
              for k, v in pb["results"]["get_in"].items()}).T

,auroc,ap,ap_baseline,captured,false_alarms
ens_crowd9,0.551,0.137,0.111,70.0,232.0
ens_xp13,0.572,0.157,0.111,72.0,256.0
logit_xp13,0.575,0.165,0.111,68.0,292.0
gbm_xp13,0.548,0.136,0.111,73.0,266.0


## 5. Operating point and alert shape

In [16]:
pd.DataFrame(rec("operating_point_sweep"))[
    ["head", "beta", "capture_rate", "precision", "fa_per_iy", "auroc", "ap"]]

,head,beta,capture_rate,precision,fa_per_iy,auroc,ap
0,get_out,0.500,0.344,0.371,0.351,0.731,0.260
1,get_out,0.625,0.360,0.375,0.361,0.731,0.260
2,get_out,0.750,0.390,0.353,0.429,0.731,0.260
3,get_out,0.875,0.419,0.342,0.483,0.731,0.260
4,get_out,1.000,0.445,0.317,0.575,0.731,0.260
5,get_in,0.500,0.391,0.575,0.172,0.734,0.257
6,get_in,0.625,0.444,0.504,0.259,0.734,0.257
7,get_in,0.750,0.526,0.503,0.308,0.734,0.257
8,get_in,0.875,0.526,0.503,0.308,0.734,0.257
9,get_in,1.000,0.569,0.455,0.404,0.734,0.257


In [17]:
cs = rec("conditioning_sweep")
print("adoption rule:", cs["adoption_rule"])
pd.DataFrame({k: v for k, v in cs["results"]["GET OUT"].items()}).T

adoption rule: fewest flips among configs cutting FAs with capture within 5pts of shipped, per head; one family both heads


,captured,detectable,capture_rate,false_alarms,fa_per_iy,precision,episode_f1,n_alerts,flips_within_30d
raw|F1.0|level,93.0,218.0,0.427,165.0,0.346,0.360,0.391,305.0,241.0
raw|F1.0|crossing,89.0,218.0,0.408,154.0,0.323,0.366,0.386,271.0,183.0
raw|F0.5|level,65.0,218.0,0.298,97.0,0.203,0.401,0.342,184.0,98.0
raw|F0.5|crossing,65.0,218.0,0.298,87.0,0.182,0.428,0.351,169.0,87.0
smooth|F1.0|level,76.0,218.0,0.349,150.0,0.314,0.336,0.342,255.0,264.0
smooth|F1.0|crossing,71.0,218.0,0.326,117.0,0.245,0.378,0.350,196.0,146.0
smooth|F0.5|level,52.0,218.0,0.239,72.0,0.151,0.419,0.304,141.0,96.0
smooth|F0.5|crossing,49.0,218.0,0.225,63.0,0.132,0.438,0.297,116.0,65.0


In [18]:
cfg = rec("nb06_desk_config")
print("adopted:", cfg["adopted"])
pd.DataFrame(cfg["table"])

adopted: {'get_out': 'CUT EXPOSURE boom-gated SMOOTHED (production)', 'get_in': 'INCREASE EXPOSURE phase-aware SMOOTHED (production)'}


,variant,captured,detectable,late,FA,AP,adjacency
0,CUT EXPOSURE boom-gated raw,26,122,-,41,0.435,-
1,CUT EXPOSURE boom-gated SMOOTHED (production),24,122,-,39,0.449,-
2,INCREASE EXPOSURE baseline raw,29,125,21,169,0.098,20
3,INCREASE EXPOSURE baseline smoothed,28,125,18,133,0.100,18
4,INCREASE EXPOSURE phase-aware raw,24,125,12,166,0.084,8
5,INCREASE EXPOSURE phase-aware SMOOTHED (produc...,20,125,10,124,0.084,2


In [19]:
pd.Series({
    "EUPHORIA_FA_BUDGET_PER_IY": C.EUPHORIA_FA_BUDGET_PER_IY,
    "EUPHORIA_FA_PENALTY": C.EUPHORIA_FA_PENALTY,
    "EUPHORIA_STRICT_BETA": C.EUPHORIA_STRICT_BETA,
    "EUPHORIA_COOLDOWN_DAYS": C.EUPHORIA_COOLDOWN_DAYS,
    "EUPHORIA_ALERT_SPACING_D": C.EUPHORIA_ALERT_SPACING_D,
    "HOLD_DAYS": C.HOLD_DAYS,
}, name="value")

EUPHORIA_FA_BUDGET_PER_IY     0.23
EUPHORIA_FA_PENALTY           1.00
EUPHORIA_STRICT_BETA          0.50
EUPHORIA_COOLDOWN_DAYS       21.00
EUPHORIA_ALERT_SPACING_D     63.00
HOLD_DAYS                    20.00
Name: value, dtype: float64

## 6. Inflection marker

In [20]:
inf = rec("inflection_trigger_sweep")
print("label:", inf["label"])
pd.DataFrame({"adopted": {"config": inf["adopted"]["config"],
                          "flags": inf["adopted"]["flags"],
                          "hit_rate": inf["adopted"]["hit_rate"],
                          "lift": inf["adopted"]["lift"]},
              "inherited": {"config": inf["inherited_from_euphoria_heads"]["config"],
                            "flags": inf["inherited_from_euphoria_heads"]["flags"],
                            "hit_rate": inf["inherited_from_euphoria_heads"]["hit_rate"],
                            "lift": inf["inherited_from_euphoria_heads"]["lift"]}})

label: {'extremum_half_window_d': 21, 'move_away_min_excess': 0.08, 'lookahead_d': 10, 'horizon_d': 21}


,adopted,inherited
config,"{'cut': 0.95, 'rearm': 0.5, 'spacing': 63}","{'cut': 0.97, 'rearm': 0.5, 'spacing': 21}"
flags,230,197
hit_rate,0.0956,0.0815
lift,1.715,1.461


In [21]:
pd.DataFrame(inf["adopted"]["by_year"])

,year,flags,hit,base
0,2019,64,0.031,0.020
1,2020,47,0.149,0.068
2,2021,48,0.188,0.072
3,2022,38,0.026,0.061
4,2023,9,0.000,0.049
5,2024,4,0.000,0.000
6,2025,3,0.000,0.000
7,2026,17,0.176,0.096


## 7. Display: the gauge zones

In [22]:
gz = rec("gauge_zones")
print(gz["outcome"])
print("amber edge:", gz["amber_edge"], "| red edge:", gz["red_edge"],
      "| base rate:", round(gz["base_rate"], 3))
pd.DataFrame(gz["edge_grid"])

a fall of 10% or more over 7 days STARTS within the next 30 days
amber edge: 76 | red edge: 85 | base rate: 0.233


,cut,seed0,seed1,seed2,seed3,seed4,all_seeds_positive
0,68,-0.014659,-0.015063,-0.015807,-0.015631,-0.014405,False
1,70,-0.009330,-0.010016,-0.011081,-0.009888,-0.009171,False
2,72,-0.004685,-0.004640,-0.005467,-0.004826,-0.003892,False
3,74,-0.000414,0.000877,-0.001198,-0.000658,0.000392,False
4,76,0.004482,0.005956,0.004913,0.003438,0.004611,True
5,78,0.010717,0.011930,0.009574,0.009742,0.009680,True
6,80,0.014096,0.015454,0.012788,0.013122,0.012682,True
7,82,0.019516,0.021945,0.018990,0.019989,0.018867,True
8,84,0.016943,0.019633,0.016642,0.016451,0.016966,True
9,86,0.019464,0.022362,0.017889,0.018178,0.019913,True


## 8. Bot screen

The settings and the last stored run, if any.

In [23]:
from src import settings                                        # noqa: E402

pd.Series({k: settings.get(k) for k in (
    "bot_screen_enabled", "bot_screen_threshold",
    "bot_screen_duplicate_jaccard", "bot_screen_burst_posts_per_day")},
    name="value")

bot_screen_enabled                true
bot_screen_threshold               0.6
bot_screen_duplicate_jaccard      0.85
bot_screen_burst_posts_per_day      12
Name: value, dtype: str

In [24]:
last = os.path.join(ROOT, "data", "reference", "bot_screen_last.json")
if os.path.exists(last):
    with open(last, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=1))
else:
    print("no bot-screen report stored yet (runs with the next refresh)")

no bot-screen report stored yet (runs with the next refresh)
